In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import os
import numpy as np

base_path = "/kaggle/input/q1-stage-3-2026"

dir_contents = os.listdir(base_path)
print(f"Contents of {base_path}: {dir_contents}")

if dir_contents:
    actual_dataset_folder = None
    for item in dir_contents:
        if os.path.isdir(os.path.join(base_path, item)) and item not in ['.', '..']:
            actual_dataset_folder = item
            break

    if actual_dataset_folder:
        path = os.path.join(base_path, actual_dataset_folder)
        print(f"Adjusted dataset path to: {path}")
    else:
        raise FileNotFoundError(f"No suitable dataset subfolder found in {base_path}. Contents: {dir_contents}")
else:
    raise FileNotFoundError(f"Base dataset directory {base_path} is empty.")


train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=path, transform=train_transform)
test_dataset = datasets.ImageFolder(root=path, transform=test_transform)

full_size = len(train_dataset)
train_size = int(0.8 * full_size)
test_size = full_size - train_size

indices = torch.randperm(full_size).tolist()
train_indices = indices[:train_size]
test_indices = indices[train_size:]

train_data_subset = torch.utils.data.Subset(train_dataset, train_indices)
test_data_subset = torch.utils.data.Subset(test_dataset, test_indices)

train_loader = DataLoader(train_data_subset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data_subset, batch_size=32, shuffle=False)

images, labels = next(iter(train_loader))
classes = train_dataset.classes

plt.figure(figsize=(10, 5))
for i in range(5):
    plt.subplot(1, 5, i+1)
    img = images[i].permute(1, 2, 0).numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img * std + mean
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    plt.title(classes[labels[i]])
    plt.axis('off')
plt.show()


In [ ]:
# Write your code here
class PotatoCNN(nn.Module):
    def __init__(self):
        super(PotatoCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Linear(512, 3)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


In [ ]:
# Write your code here
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return loss_sum/len(loader), 100*correct/len(loader.dataset)

def validate_epoch(model, loader, criterion, device):
    model.eval()
    loss_sum, correct = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss_sum += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
    return loss_sum/len(loader), 100*correct/len(loader.dataset)


In [ ]:
# Write your code here
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PotatoCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_losses, val_losses, train_accs, val_accs = [], [], [], []
for epoch in range(10):
    t_loss, t_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    v_loss, v_acc = validate_epoch(model, test_loader, criterion, device)
    train_losses.append(t_loss); val_losses.append(v_loss)
    train_accs.append(t_acc); val_accs.append(v_acc)
    print(f"Epoch {epoch+1}: Val Acc {v_acc:.2f}%")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train'); plt.plot(val_losses, label='Val'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train'); plt.plot(val_accs, label='Val'); plt.legend()
plt.show()

In [ ]:
# Write your code here

class PotatoResidualCNN(nn.Module):
    def __init__(self):
        super(PotatoResidualCNN, self).__init__()
        self.c1 = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.c2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.c3 = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.c4 = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.c5 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Linear(256, 3))

    def forward(self, x):
        x = self.c1(x)
        skip = self.c2(x)
        x = self.c3(skip)
        x = self.c4(x) + skip
        return self.fc(self.c5(x))

res_model = PotatoResidualCNN().to(device)
optimizer_res = optim.Adam(res_model.parameters(), lr=0.001)
for epoch in range(10):
    _, acc = train_epoch(res_model, train_loader, criterion, optimizer_res, device)
    _, v_acc = validate_epoch(res_model, test_loader, criterion, device)
    print(f"Res Epoch {epoch+1}: Val Acc {v_acc:.2f}%")